# 02 — Statistical Inference (Task 4)

Brief section 5. For every test: state H0/H1, justify the test (incl. assumption
checks), report the statistic/df/p-value, report an effect size, interpret, and
state the practical implication for management.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg

clean = pd.read_csv('../data/processed/invoice_lines_clean.csv', parse_dates=['InvoiceDate'])
sales = clean[(~clean['IsCancellation']) & (clean['Quantity'] > 0)].copy()
customers = pd.read_csv('../data/processed/customer_table.csv')

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

In [2]:
from scipy import stats

## Test 1 — Comparison of means
**Question:** Do international customers have a higher average order value than UK customers?

**Method:** Welch t-test on log order value; Mann–Whitney U as a robustness check.

In [3]:
import numpy as np

uk = customers.loc[customers.IsUK == 1, 'AvgBasketValue']
intl = customers.loc[customers.IsUK == 0, 'AvgBasketValue']

log_uk, log_intl = np.log1p(uk), np.log1p(intl)

# assumption check
lev_stat, lev_p = stats.levene(log_uk, log_intl, center='median')

# main test
t_stat, t_p = stats.ttest_ind(log_intl, log_uk, equal_var=False)  # Welch

# robustness check
u_stat, u_p = stats.mannwhitneyu(intl, uk, alternative='greater')

# effect size: Cohen's d on the log scale
pooled_sd = np.sqrt((log_uk.var(ddof=1) + log_intl.var(ddof=1)) / 2)
cohens_d = (log_intl.mean() - log_uk.mean()) / pooled_sd

print(f"Levene (log): stat={lev_stat:.3f}, p={lev_p:.4f}")
print(f"Welch t-test: t={t_stat:.3f}, p={t_p:.4f}")
print(f"Mann-Whitney U: U={u_stat:.1f}, p={u_p:.4f}")
print(f"Cohen's d (log scale): {cohens_d:.3f}")

Levene (log): stat=35.417, p=0.0000
Welch t-test: t=12.402, p=0.0000
Mann-Whitney U: U=1495183.0, p=0.0000
Cohen's d (log scale): 0.656


## Test 2 — Comparison of proportions
**Question:** Does the repurchase rate differ for Q4-acquired customers?

**Method:** Two-proportion z-test or chi-square test.

In [4]:
first_purchase = (clean[clean.InvoiceDate < '2011-09-09']
                   .groupby('CustomerID')['InvoiceDate'].min())

q4_flag = first_purchase.dt.month.isin([10, 11, 12])

customers['AcquiredInQ4'] = customers['CustomerID'].map(q4_flag).fillna(False)

In [5]:
ct = pd.crosstab(customers.AcquiredInQ4, customers.Repurchase)
chi2, p, dof, expected = stats.chi2_contingency(ct)

# check expected-cell-count assumption
print((expected < 5).sum(), 'cells below 5')  # chi-square needs this checked

# effect size: Cramer's V
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

0 cells below 5


In [6]:
print(f"Chi-square: χ²={chi2:.3f}, dof={dof}, p={p:.4f}")
print(f"Cramér's V: {cramers_v:.3f}")

Chi-square: χ²=47.321, dof=1, p=0.0000
Cramér's V: 0.095


## Test 3 — Comparison of variances
**Question:** Is order value more variable for international customers?

**Method:** Levene or Brown–Forsythe test (F-test too sensitive to non-normality).

In [7]:
lev_stat2, lev_p2 = stats.levene(uk, intl, center='median')  # Brown-Forsythe

# effect size-ish: variance ratio
var_ratio = intl.var(ddof=1) / uk.var(ddof=1)

In [8]:
print(f"Brown-Forsythe (Levene, median-centered): stat={lev_stat2:.3f}, p={lev_p2:.4f}")
print(f"Variance ratio (Intl/UK): {var_ratio:.2f}")

Brown-Forsythe (Levene, median-centered): stat=123.940, p=0.0000
Variance ratio (Intl/UK): 5.24


## Test 4 — ANOVA
**Question:** Does basket value differ by day of week or RFM segment?

**Method:** Welch ANOVA with Games-Howell post-hoc; Kruskal-Wallis as a check.

In [9]:
sales['dow'] = sales['InvoiceDate'].dt.day_name()

In [10]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
sales['dow'] = pd.Categorical(sales['dow'], categories=day_order, ordered=True)

In [11]:
basket = sales.groupby(['Invoice','InvoiceDate'])['LineRevenue'].sum().reset_index()
basket['dow'] = pd.Categorical(basket['InvoiceDate'].dt.day_name(), categories=day_order, ordered=True)
groups_dow = [g['LineRevenue'].values for _, g in basket.groupby('dow')]

In [12]:
aov_dow = pg.welch_anova(dv='LineRevenue', between='dow', data=basket)
gh_dow = pg.pairwise_gameshowell(dv='LineRevenue', between='dow', data=basket)
kw_dow_stat, kw_dow_p = stats.kruskal(*groups_dow)

print(aov_dow)
print(f"Kruskal-Wallis (day of week): H={kw_dow_stat:.3f}, p={kw_dow_p:.4f}")

  Source  ddof1       ddof2          F         p_unc       np2
0    dow      6  466.647762  32.403064  1.243826e-32  0.002181
Kruskal-Wallis (day of week): H=63.907, p=0.0000


In [14]:
customers['R_q'] = pd.qcut(-customers.Recency, 4, labels=False, duplicates='drop')
customers['F_q'] = pd.qcut(customers.Frequency.rank(method='first'), 4, labels=False)
customers['M_q'] = pd.qcut(customers.Monetary, 4, labels=False)
customers['RFM_score'] = customers.R_q + customers.F_q + customers.M_q
customers['RFM_segment'] = pd.cut(customers.RFM_score, bins=[-1,2,5,9],
                                   labels=['Low','Mid','High'])

In [15]:
aov = pg.welch_anova(dv='AvgBasketValue', between='RFM_segment', data=customers)
gh = pg.pairwise_gameshowell(dv='AvgBasketValue', between='RFM_segment', data=customers)

kw_stat, kw_p = stats.kruskal(*[g['AvgBasketValue'].values
                                 for _, g in customers.groupby('RFM_segment')])

print(aov)
print(gh)
print(f"Kruskal-Wallis (RFM segment): H={kw_stat:.3f}, p={kw_p:.4f}")

        Source  ddof1        ddof2           F         p_unc       np2
0  RFM_segment      2  2932.480895  138.432824  3.553145e-58  0.025376
     A     B      mean_A      mean_B        diff         se          T  \
0  Low   Mid  246.322124  381.032143 -134.710019  16.572559  -8.128498   
1  Low  High  246.322124  439.073085 -192.750961  12.541060 -15.369591   
2  Mid  High  381.032143  439.073085  -58.040942  19.582188  -2.963966   

            df          pval    hedges  
0  1986.779325  1.079137e-12 -0.278791  
1  2687.282967  0.000000e+00 -0.472852  
2  3167.127588  8.593335e-03 -0.100194  
Kruskal-Wallis (RFM segment): H=619.976, p=0.0000


## Key points to note in write-up

- Monetary variables are right-skewed — justify log transforms / non-parametric pairing.
- With thousands of customers, almost everything is significant — effect sizes are essential.